# NYC Heartbeat: The Pulse of a City Through its Bikes

**Course:** 02806 Social Data Analysis and Visualization  
**Dataset:** Citi Bike NYC Trip Data (2025)  
**Author:** Marcus Lilleør Christoffersen (s224750)

---

This notebook contains the behind-the-scenes analysis for the final project. 

**Sections:**
1. [Motivation](#1-motivation)
2. [Basic Stats](#2-basic-stats)
3. [Data Analysis](#3-data-analysis)
4. [Genre](#4-genre)
5. [Visualizations](#5-visualizations)
6. [Discussion](#6-discussion)
7. [Contributions](#7-contributions)

## 1. Motivation

**What is the dataset?**
I use the [Citi Bike NYC trip data](https://citibikenyc.com/system-data) for the full year 2025: 12 monthly ZIP files, ~8.5 GB of raw CSVs, over 45 million rides across 2,344 unique stations. For analysis I work with a representative sample of ~12 million trips (one CSV file per month), which preserves the seasonal distribution while keeping the data tractable.

**Why this dataset?**
Bike-share trips are a proxy for human movement at city scale. Each trip has a precise origin, destination, timestamp, rider type, and bike type, giving a complete view of how and when the city moves. The full year allows observation of seasonal patterns and anomalies like storms or holidays.

**Goal for the end user's experience:**
The website is a three-act narrative. Act 1 shows how the city moves hour by hour. Act 2 shows how it organizes into functional neighborhoods. Act 3 shows when it breaks its pattern. The central question: *what does NYC's heartbeat actually look like, and what disrupts it?*

## 2. Basic Stats

### Data Cleaning

The raw data requires minimal cleaning. Key choices:

- **Columns dropped:** `ride_id`, `start_station_id`, `end_station_id` (station names are more readable and equivalent).
- **Rows dropped:** Missing station names (~2% of rows), trips under 1 minute (docking errors) or over 24 hours (unreturned bikes).
- **Sampling:** One CSV file per month was selected and concatenated into a single `.parquet` file (~12 million rows). This keeps the full seasonal and weekday distribution while reducing memory requirements.
- **Derived columns:** `duration_min`, `hour`, `dayofweek`, `month`, `is_weekend`.

No imputation was performed. Rows with missing coordinates were dropped since they are required for geographic visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

PARQUET_URL = "https://github.com/Pakkutaq/socialdata2026/releases/download/v1.0-data/citibike_2025_sampled.parquet"

df = pd.read_parquet(PARQUET_URL)

df["started_at"]   = pd.to_datetime(df["started_at"])
df["ended_at"]     = pd.to_datetime(df["ended_at"])
df["duration_min"] = (df["ended_at"] - df["started_at"]).dt.total_seconds() / 60
df["hour"]         = df["started_at"].dt.hour
df["dayofweek"]    = df["started_at"].dt.dayofweek
df["month"]        = df["started_at"].dt.month
df["is_weekend"]   = df["dayofweek"] >= 5

df = df.dropna(subset=["start_station_name", "end_station_name",
                        "start_lat", "start_lng", "end_lat", "end_lng"])
df = df[(df["duration_min"] >= 1) & (df["duration_min"] <= 1440)]

print(f"Rows:              {len(df):>12,}")
print(f"Unique stations:   {df['start_station_name'].nunique():>12,}")
print(f"Date range:        {df['started_at'].min().date()} -> {df['started_at'].max().date()}")
print(f"Member rides:      {(df['member_casual']=='member').mean()*100:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), facecolor="#1a1a2e")
fmt = mticker.FuncFormatter(lambda x, _: f"{int(x):,}")

n_weekdays = df[~df["is_weekend"]]["started_at"].dt.date.nunique()
n_weekend_days = df[df["is_weekend"]]["started_at"].dt.date.nunique()

# 1. Trips by hour
ax = axes[0, 0]
ax.set_facecolor("#1a1a2e")
hourly = df.groupby("hour").size()
ax.bar(hourly.index, hourly.values, color="#4FC3F7", edgecolor="#1a1a2e")
ax.set_title("Trips by Hour of Day", color="white")
ax.set_xlabel("Hour", color="white"); ax.set_ylabel("Trips", color="white")
ax.tick_params(colors="white"); ax.yaxis.set_major_formatter(fmt)
for s in ax.spines.values(): s.set_color("#444")

# 2. Weekday vs Weekend hourly overlay
ax = axes[0, 1]
ax.set_facecolor("#1a1a2e")
wd = df[~df["is_weekend"]].groupby("hour").size() / n_weekdays
we = df[df["is_weekend"]].groupby("hour").size() / n_weekend_days
ax.plot(wd.index, wd.values, color="#4FC3F7", label="Weekday avg")
ax.plot(we.index, we.values, color="#FF8A65", label="Weekend avg")
ax.set_title("Weekday vs Weekend (avg daily trips)", color="white")
ax.set_xlabel("Hour", color="white"); ax.set_ylabel("Trips", color="white")
ax.tick_params(colors="white"); ax.yaxis.set_major_formatter(fmt)
ax.legend(facecolor="#2a2a3e", edgecolor="#555", labelcolor="white")
for s in ax.spines.values(): s.set_color("#444")

# 3. Trips by month
ax = axes[1, 0]
ax.set_facecolor("#1a1a2e")
monthly = df.groupby("month").size()
months = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
ax.bar(range(1, 13), monthly.values, color="#4FC3F7", edgecolor="#1a1a2e")
ax.set_xticks(range(1, 13)); ax.set_xticklabels(months, rotation=45)
ax.set_title("Trips by Month", color="white")
ax.set_xlabel("Month", color="white"); ax.set_ylabel("Trips", color="white")
ax.tick_params(colors="white"); ax.yaxis.set_major_formatter(fmt)
for s in ax.spines.values(): s.set_color("#444")

# 4. Trip duration distribution
ax = axes[1, 1]
ax.set_facecolor("#1a1a2e")
ax.hist(df["duration_min"].clip(upper=60), bins=60, color="#4FC3F7", edgecolor="#1a1a2e")
ax.set_title("Trip Duration (capped at 60 min)", color="white")
ax.set_xlabel("Minutes", color="white"); ax.set_ylabel("Trips", color="white")
ax.tick_params(colors="white"); ax.yaxis.set_major_formatter(fmt)
for s in ax.spines.values(): s.set_color("#444")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, facecolor="#1a1a2e", bbox_inches="tight")
plt.show()

The hourly distribution shows a clear double-peak commuter pattern, a sharp morning spike around 8am and an even larger evening spike around 5–6pm, with a midday pause between them; weekdays are strongly commute-driven while weekends flatten into a broader midday plateau. Monthly volume rises steadily from the cold winter months, peaks in July–September, then falls back in autumn, confirming that weather is the dominant seasonal driver. Trip durations are heavily right-skewed: the median is around 8–9 minutes, indicating that most rides are short utilitarian trips rather than leisure outings, even in summer.

## 3. Data Analysis

### 3.1 Flow Network

I model the network as a directed weighted graph: nodes are stations, edges are trips. For the flow map I pre-compute the top 100 station-to-station flows for each of the 24 hours (weekdays only).

The flow map consistently points to the same two areas regardless of the hour: central Midtown and the Financial District at the southern tip of Manhattan. This is not surprising once you think about it. These are the two largest employment concentrations in the city, and the Citi Bike network is, at its core, commuter infrastructure built around them. The sheer volume of trips into and out of these areas at peak hours dwarfs everything else on the map, which suggests that while recreational cycling is part of the picture, it is secondary to getting people to and from work.

What is more interesting is the shape of the daily pattern. It is not a clean morning-in, evening-out reversal. The same corridors that are busy at 8am are busy again at 6pm, but the mix of directions shifts rather than flips. Riders come from different directions, stop in different places, and the network briefly reaches its peak complexity before quieting through the night.

### 3.2 Functional Neighborhoods (Community Detection)

Louvain community detection (Blondel et al., 2008) is run on the aggregated station graph via `networkx.community.louvain_communities`, with edges weighted by total trip volume. Only station pairs with at least 50 shared trips are included; this threshold removes one-off trips that would add noise without contributing structural information, while preserving the high-frequency corridors that define each community.

The result is revealing precisely because it ignores geography: the algorithm only sees trip volume between stations, not where they are on a map. And yet it produces clusters with a clear spatial logic. The most striking finding is that stations in DUMBO and around the Brooklyn Bridge end up in the same community as stations in the Financial District. Not because they are close on the map, but because thousands of riders cross the East River between them every day.

### 3.3 Anomaly Detection

For each day in 2025 I compute total trips and compare to a rolling baseline: the mean of the same weekday over the previous four weeks. The deviation is expressed in standard deviations of the residuals (observed minus baseline), then plotted as a calendar heatmap. Days more than 1.5 sigma below baseline almost always correspond to weather events or public holidays. Since the analysis uses one CSV file per month rather than the full dataset, daily counts represent a consistent sample rather than total ridership, so the relative deviations remain valid.

In [15]:
import plotly.graph_objects as go
import plotly.express as px

weekday_df = df[~df["is_weekend"]].copy()

hourly_flows = {}
for hour in range(24):
    h = weekday_df[weekday_df["hour"] == hour]
    flows = (
        h.groupby([
            "start_station_name", "start_lat", "start_lng",
            "end_station_name",   "end_lat",   "end_lng"
        ])
        .size()
        .reset_index(name="trips")
        .sort_values("trips", ascending=False)
        .head(100)
    )
    hourly_flows[hour] = flows

print("Hourly flow snapshots ready (top 100 per hour).")
for h in [0, 8, 12, 17, 22]:
    top = hourly_flows[h].iloc[0]
    print(f"  Hour {h:02d}: {top['start_station_name']} -> {top['end_station_name']} ({top['trips']:,} trips)")

KeyboardInterrupt: 

In [ ]:
import plotly.graph_objects as go

frames = []
slider_steps = []

for hour in range(24):
    flows = hourly_flows[hour].copy()

    stations = (
        weekday_df[weekday_df["hour"] == hour]
        .groupby(["start_station_name", "start_lat", "start_lng"])
        .size()
        .reset_index(name="trips")
    )
    stations["size"] = stations["trips"] / stations["trips"].max() * 15 + 3

    lons, lats = [], []
    for _, row in flows.iterrows():
        lons += [row["start_lng"], row["end_lng"], None]
        lats += [row["start_lat"], row["end_lat"], None]

    frame_traces = [
        go.Scattermap(
            lat=stations["start_lat"], lon=stations["start_lng"],
            mode="markers",
            marker=dict(
                size=stations["size"],
                color=stations["trips"],
                colorscale="Plasma",
                showscale=True,
                colorbar=dict(title="Trips", tickfont=dict(color="white"))
            ),
            text=stations["start_station_name"],
            hovertemplate="<b>%{text}</b><br>Trips: %{marker.color}<extra></extra>",
            showlegend=False
        ),
        go.Scattermap(
            lon=lons, lat=lats,
            mode="lines",
            line=dict(width=2, color="rgba(79,195,247,0.5)"),
            hoverinfo="skip", showlegend=False
        ),
    ]

    frames.append(go.Frame(data=frame_traces, name=str(hour)))
    slider_steps.append(dict(
        args=[[str(hour)], {"frame": {"duration": 0}, "mode": "immediate"}],
        label=f"{hour:02d}:00",
        method="animate"
    ))

fig = go.Figure(data=frames[8].data, frames=frames)
fig.update_layout(
    map=dict(style="carto-darkmatter", center={"lat": 40.74, "lon": -73.99}, zoom=12),
    paper_bgcolor="#1a1a2e", font_color="white",
    title="NYC Bike Flow by Hour (weekdays)",
    height=650, margin=dict(l=0, r=0, t=40, b=0),
    updatemenus=[],
    sliders=[dict(
        active=8, steps=slider_steps,
        x=0, y=0, len=1.0,
        currentvalue=dict(prefix="Hour: ", font=dict(color="white")),
        bgcolor="#2a2a3e", bordercolor="#444",
        font=dict(color="white")
    )]
)
fig.show()

In [ ]:
fig.write_html("viz1_flow.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
import networkx as nx

# --- 3.2 Community detection ---
edges = (
    df.groupby(["start_station_name", "end_station_name"])
    .size()
    .reset_index(name="weight")
)
edges = edges[edges["start_station_name"] != edges["end_station_name"]]
edges = edges[edges["weight"] >= 50]

G = nx.from_pandas_edgelist(
    edges,
    source="start_station_name",
    target="end_station_name",
    edge_attr="weight"
)

communities = nx.community.louvain_communities(G, weight="weight", seed=42)
partition = {node: i for i, comm in enumerate(communities) for node in comm}

station_coords = (
    df.groupby("start_station_name")[["start_lat", "start_lng"]]
    .mean()
    .reset_index()
    .rename(columns={"start_station_name": "station", "start_lat": "lat", "start_lng": "lng"})
)
station_coords["community"] = station_coords["station"].map(partition)
station_coords = station_coords.dropna(subset=["community"])
station_coords["community"] = station_coords["community"].astype(str)

top_communities = station_coords["community"].value_counts().head(8).index
station_coords = station_coords[station_coords["community"].isin(top_communities)]

print(f"Nodes in graph: {G.number_of_nodes():,}")
print(f"Edges in graph: {G.number_of_edges():,}")
print(f"Communities found: {len(communities)}")
print(f"Top 8 communities cover {len(station_coords)} stations")

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    station_coords, lat="lat", lon="lng", color="community",
    hover_name="station", zoom=11,
    center={"lat": 40.74, "lon": -73.99},
    map_style="dark",
    color_discrete_sequence=px.colors.qualitative.Bold,
    title="Functional Neighborhoods (Louvain Communities)",
    height=620
)
fig.update_traces(marker=dict(size=8, opacity=0.85))
fig.update_layout(paper_bgcolor="#1a1a2e", font_color="white",
                  legend_title="Community", margin=dict(l=0, r=0, t=40, b=0))
fig.show()

In [ ]:
fig.write_html("viz2_community.html", include_plotlyjs="cdn", full_html=True)

In [ ]:
import numpy as np

daily = (
    df.groupby(df["started_at"].dt.date)
    .size()
    .reset_index(name="trips")
)
daily.columns = ["date", "trips"]
daily["date"] = pd.to_datetime(daily["date"])
daily["weekday"] = daily["date"].dt.weekday
daily = daily.sort_values("date").reset_index(drop=True)

baselines = []
for _, row in daily.iterrows():
    same_weekday = daily[
        (daily["weekday"] == row["weekday"]) & (daily["date"] < row["date"])
    ].tail(4)["trips"]
    baselines.append(same_weekday.mean() if len(same_weekday) >= 2 else np.nan)

daily["baseline"] = baselines
residuals = daily["trips"] - daily["baseline"]
daily["deviation"] = residuals / residuals.std()

print(f"Days covered: {len(daily)}")
print(daily.dropna(subset=["deviation"]).nsmallest(5, "deviation")[["date","trips","deviation"]].to_string(index=False))

In [ ]:
import plotly.graph_objects as go

daily_clean = daily.dropna(subset=["deviation"]).copy()
daily_clean["week"] = daily_clean["date"].dt.isocalendar().week.astype(int)
daily_clean["weekday"] = daily_clean["date"].dt.weekday
daily_clean["label"] = daily_clean["date"].dt.strftime("%b %d")

fig = go.Figure(go.Heatmap(
    x=daily_clean["week"],
    y=daily_clean["weekday"],
    z=daily_clean["deviation"],
    text=daily_clean["label"],
    hovertemplate="<b>%{text}</b><br>Deviation: %{z:.2f}σ<extra></extra>",
    colorscale="RdYlGn", zmid=0,
    colorbar=dict(title="Deviation (σ)", tickfont=dict(color="white"))
))

fig.update_layout(
    title="When Does the City Break Its Pattern?",
    paper_bgcolor="#1a1a2e", plot_bgcolor="#1a1a2e", font_color="white",
    yaxis=dict(tickvals=[0,1,2,3,4,5,6],
               ticktext=["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]),
    xaxis_title="Week of year",
    height=420, margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

In [ ]:
fig.write_html("viz3_calendar.html", include_plotlyjs="cdn", full_html=True)

## 4. Genre

I chose the **magazine** genre (Segel & Heer, 2010, Figure 7). The data has a natural three-act arc where each act builds on the previous, so linear reading order matters. A magazine layout enforces this without being as rigid as a slideshow, and without the open-endedness of a pure interactive tool that can leave readers without a narrative frame.

The table below maps each Figure 7 tool to its concrete use in this project.

### Visual Narrative

| Category | Tool used | Where |
|---|---|---|
| **Visual Structuring** | Consistent visual platform | All three visualisations share the same dark map style (`carto-darkmatter`) and colour palette, so the reader's eye never has to re-orient between acts. |
| **Highlighting** | Hover tooltips | Station names and trip counts appear on hover in Viz 1; community names on hover in Viz 2; date and deviation in Viz 3. Detail is surfaced on demand so the base view stays uncluttered. |
| **Transition Guidance** | Animated slider | The 24-hour time slider in Viz 1 gives the reader explicit control over the transition between states, making the change between hours feel like a continuous animation rather than a jump. |

### Narrative Structure

| Category | Tool used | Where |
|---|---|---|
| **Ordering** | Linear, author-driven | The three acts are sequenced deliberately: movement, organisation, disruption. Each act depends conceptually on the previous, so the reader cannot skip ahead without losing context. |
| **Interactivity** | Hover + slider | Viz 1 allows the reader to scrub through hours at their own pace; all three maps support hover. The level of interactivity is limited to exploration within a pre-defined frame, consistent with the magazine genre. |
| **Messaging** | Introductory text + captions | Each section opens with a paragraph that primes the reader on what to look for before the visualisation is shown, and closes with a paragraph that draws the analytical conclusion. Captions below each visualisation describe the data source and methodological choices. |

## 5. Visualizations

### Viz 1: Flow Network with Time Slider
Interactive Plotly map of the top 100 station-to-station flows per hour (weekdays). All flow lines are rendered as a single trace for performance; station dots are sized and colored by total departures in the selected hour. A slider scrubs through 24 hours.

**Why:** Flow lines on a map are the clearest way to show which corridors carry the most volume. A chord diagram or OD matrix would lose the spatial intuition of which neighborhoods are sources vs. sinks.

### Viz 2: Community Map
Plotly scatter map with each station colored by its Louvain community (top 8 shown).

**Why:** Color-encoded geographic scatter is the clearest way to show that functional neighborhoods do not follow the official borough map.

### Viz 3: Calendar Heatmap
Plotly heatmap with weeks on the x-axis and weekdays on the y-axis, colored by deviation from the rolling same-weekday baseline (green = above, red = below).

**Why:** A calendar heatmap makes it easy to compare the same weekday across weeks, which is exactly what the anomaly baseline requires. A line chart would not.

## 6. Discussion

**What went well?**
The three-act structure maps cleanly onto the data. The flow network shows how activity concentrates around Midtown and the Financial District and how the corridor mix shifts across the day. The Louvain communities produce coherent geographic clusters that clearly cross borough boundaries.

**What could be improved?**
- The flow map shows weekday averages only. A member vs. casual split would reveal that commuters and tourists have very different spatial patterns.
- Encoding all 24 hours of flow data into a single self-contained HTML file results in a large file (~3.5 MB). This works well as a visual for the website but would not scale to showing all trips rather than the top 100 per hour. A server-side approach with on-demand data loading would be needed to go beyond that.
- The anomaly baseline is simple (rolling 4-week mean). STL decomposition would separate trend, seasonality, and residuals more robustly.
- Community detection uses an undirected graph. A directed version could reveal stations that are net exporters in the morning and importers in the evening.
- Joining weather data would let me quantitatively validate the storm/holiday anomaly hypothesis.

**References:**
- Segel, E. and Heer, J. (2010). Narrative Visualization: Telling Stories with Data. *IEEE TVCG*, 16(6), 1139-1148.
- Blondel, V. D. et al. (2008). Fast unfolding of communities in large networks. *J. Stat. Mech.*, P10008.
- Citi Bike System Data: https://citibikenyc.com/system-data

## 7. Contributions

This was a solo project. All work was completed by **Marcus Lilleør Christoffersen** (s224750).